In [7]:
import os
import sys
import logging
from pathlib import Path
from dotenv import load_dotenv
import torch
from torch import nn
import pandas as pd
from huggingface_hub import login

logging.basicConfig(
    level=logging.INFO,
    format="%(name)s | %(levelname)s | %(message)s",
)

torch.manual_seed(123)

# src_path = Path.cwd().parent / "src"
# if src_path.exists() and str(src_path) not in sys.path:
#     sys.path.insert(0, str(src_path))

# print(f"Added to sys.path: {src_path}")
load_dotenv()  # reads .env file from the current directory

PATH_DATA = Path.cwd().parent.parent / ".data"
PATH_GPT2_124M_WEIGHTS = PATH_DATA / "model_weights" / "gpt2" / "124M" / "parameters.pickle.gz"


In [8]:
login(os.getenv("HF_TOKEN"))

httpx | INFO | HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
huggingface_hub._login | WARNING | Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [9]:
from datasets import load_dataset

dataset = "jtviegas/financial_phrasebank"

train_ds = load_dataset(dataset, split="train")

httpx | INFO | HTTP Request: HEAD https://huggingface.co/datasets/jtviegas/financial_phrasebank/resolve/main/README.md "HTTP/1.1 404 Not Found"
httpx | INFO | HTTP Request: GET https://huggingface.co/api/datasets/jtviegas/financial_phrasebank "HTTP/1.1 200 OK"
httpx | INFO | HTTP Request: HEAD https://huggingface.co/datasets/jtviegas/financial_phrasebank/resolve/3e83f8bf254692c4f873567632c2e43e515a4408/financial_phrasebank.py "HTTP/1.1 404 Not Found"
httpx | INFO | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/jtviegas/financial_phrasebank/jtviegas/financial_phrasebank.py "HTTP/1.1 404 Not Found"
httpx | INFO | HTTP Request: HEAD https://huggingface.co/datasets/jtviegas/financial_phrasebank/resolve/3e83f8bf254692c4f873567632c2e43e515a4408/README.md "HTTP/1.1 404 Not Found"
httpx | INFO | HTTP Request: HEAD https://huggingface.co/datasets/jtviegas/financial_phrasebank/resolve/3e83f8bf254692c4f873567632c2e43e515a4408/.huggingface.yaml "HTTP/1.1 404

In [10]:
ds = train_ds.train_test_split(test_size=0.2)
train_texts = list(ds["train"]["sentence"])
test_texts = list(ds["test"]["sentence"])
train_labels = list(ds["train"]["label"])
test_labels = list(ds["test"]["label"])

In [11]:
import tiktoken
from tgedr_lm.classifier.text_dataset import TextDataset

tokenizer = tiktoken.get_encoding("gpt2")
train_dataset = TextDataset(tokenizer=tokenizer, texts=train_texts, labels=train_labels)
val_dataset = TextDataset(tokenizer=tokenizer, texts=test_texts, labels=test_labels)

In [12]:
from tgedr_lm.classifier.gpt2.hyperparam_search import HyperParamSearch
from tgedr_lm.classifier.gpt2.model import GPT2Classifier
from tgedr_lm.configuration import ClassifierBaseConfiguration, TrainingArgs

training_args = TrainingArgs()
hp_search = HyperParamSearch(GPT2Classifier.compute_metrics, train_args=training_args.to_training_arguments())
hyperparameters = hp_search.search(model=GPT2Classifier(ClassifierBaseConfiguration(n_classes=3)), 
                                       train_dataset=train_dataset, val_dataset=val_dataset, 
                                       trials=3)


tgedr_lm.classifier.gpt2.hyperparam_search | INFO | [search|in] (GPT2Classifier(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_projection): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNormalization()
      (norm2): LayerNormalization()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
 

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F
1,0.780280,0.827363,0.652577,0.543671,0.426480,0.420955
2,0.801284,0.797458,0.621649,0.480111,0.459039,0.423976
3,0.650316,0.807270,0.674227,0.579209,0.462707,0.467681
4,0.817607,0.919262,0.617526,0.545866,0.537313,0.532837
5,0.298031,1.064752,0.691753,0.618325,0.626591,0.613297
6,0.431753,1.812009,0.690722,0.615584,0.627273,0.617575


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-07-13 02:37:04,016] Trial 2 finished with value: 0.7103092783505155 and parameters: {'learning_rate': 8.4360634494296e-05, 'weight_decay': 0.008318018844866137, 'num_train_epochs': 4, 'per_device_train_batch_size': 4, 'warmup_steps': 0.14675002360719278}. Best is trial 2 with value: 0.7103092783505155.
tgedr_lm.classifier.gpt2.hyperparam_search | INFO | [search|out] => {'learning_rate': 8.4360634494296e-05, 'weight_decay': 0.008318018844866137, 'num_train_epochs': 4, 'per_device_train_batch_size': 4, 'warmup_steps': 0.14675002360719278}


In [13]:
from tgedr_lm.configuration import TrainingArgs

training_args = TrainingArgs()
for key in [
    "learning_rate",
    "weight_decay",
    "num_train_epochs",
    "per_device_train_batch_size",
    "warmup_steps",
]:
    if key in hyperparameters:
        training_args.set(key, hyperparameters[key])
        
training_args.set("hub_model_id", "jtviegas/gpt2classifier")
model = GPT2Classifier(ClassifierBaseConfiguration(n_classes=3))

In [14]:
from transformers import Trainer

trainer = Trainer(
        model=model,
        args=training_args.to_training_arguments(),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=model.compute_metrics,
        # Optional: add callbacks for custom behavior
    )

trainer.train()
final_metrics = trainer.evaluate()

accuracy = final_metrics.get("eval_accuracy")
loss = final_metrics.get("eval_loss")
# model.save_pretrained("./final_model")
trainer.push_to_hub(tags="text-classification", commit_message="Training completed!")

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F
1,0.849034,0.817621,0.667010,0.418424,0.410138,0.381590
2,0.782716,0.786553,0.640206,0.525651,0.466709,0.466325
3,0.487546,0.846295,0.695876,0.436037,0.454160,0.432865
4,0.797786,0.878512,0.682474,0.588261,0.499533,0.499620


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

httpx | INFO | HTTP Request: POST https://huggingface.co/api/validate-yaml "HTTP/1.1 200 OK"
httpx | INFO | HTTP Request: POST https://huggingface.co/api/validate-yaml "HTTP/1.1 200 OK"
httpx | INFO | HTTP Request: POST https://huggingface.co/api/models/jtviegas/gpt2classifier/preupload/main "HTTP/1.1 200 OK"
httpx | INFO | HTTP Request: POST https://huggingface.co/jtviegas/gpt2classifier.git/info/lfs/objects/batch "HTTP/1.1 200 OK"
httpx | INFO | HTTP Request: GET https://huggingface.co/api/models/jtviegas/gpt2classifier/xet-write-token/main "HTTP/1.1 200 OK"


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

httpx | INFO | HTTP Request: POST https://huggingface.co/api/models/jtviegas/gpt2classifier/commit/main "HTTP/1.1 200 OK"


CommitInfo(commit_url='https://huggingface.co/jtviegas/gpt2classifier/commit/fcac2703a5550969f0a2a0553bb24e5a40780a06', commit_message='Training completed!', commit_description='', oid='fcac2703a5550969f0a2a0553bb24e5a40780a06', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jtviegas/gpt2classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='jtviegas/gpt2classifier'), pr_revision=None, pr_num=None)